In [1]:
# load packages
import pandas as pd

### 1. Import all data files

In [3]:
visits = pd.read_csv('visits.csv', parse_dates=[1])
cart = pd.read_csv('cart.csv', parse_dates=[1])
checkout = pd.read_csv('checkout.csv', parse_dates=[1])
purchase = pd.read_csv('purchase.csv', parse_dates=[1])

### 2. Inspect the DataFrames 

In [8]:
# inspect the first 5 rows of each DataFrame
print("Visits DataFrame:")
print(visits.head(5))

print("\nCart DataFrame:")
print(cart.head(5))

print("\nCheckout DataFrame:")
print(checkout.head(5))

print("\nPurchase DataFrame:")
print(purchase.head(5))

Visits DataFrame:
                                user_id          visit_time
0  943647ef-3682-4750-a2e1-918ba6f16188 2017-04-07 15:14:00
1  0c3a3dd0-fb64-4eac-bf84-ba069ce409f2 2017-01-26 14:24:00
2  6e0b2d60-4027-4d9a-babd-0e7d40859fb1 2017-08-20 08:23:00
3  6879527e-c5a6-4d14-b2da-50b85212b0ab 2017-11-04 18:15:00
4  a84327ff-5daa-4ba1-b789-d5b4caf81e96 2017-02-27 11:25:00

Cart DataFrame:
                                user_id           cart_time
0  2be90e7c-9cca-44e0-bcc5-124b945ff168 2017-11-07 20:45:00
1  4397f73f-1da3-4ab3-91af-762792e25973 2017-05-27 01:35:00
2  a9db3d4b-0a0a-4398-a55a-ebb2c7adf663 2017-03-04 10:38:00
3  b594862a-36c5-47d5-b818-6e9512b939b3 2017-09-27 08:22:00
4  a68a16e2-94f0-4ce8-8ce3-784af0bbb974 2017-07-26 15:48:00

Checkout DataFrame:
                                user_id       checkout_time
0  d33bdc47-4afa-45bc-b4e4-dbe948e34c0d 2017-06-25 09:29:00
1  4ac186f0-9954-4fea-8a27-c081e428e34e 2017-04-07 20:11:00
2  3c9c78a7-124a-4b77-8d2e-e1926e011e7d 2017

### 3. Left merge visits and cart

In [ ]:
visits_cart_left = pd.merge(visits, cart, how='left')

# inspect the first 5 rows of the merged DataFrame
visits_cart_left.head()

,user_id,visit_time,cart_time
0,943647ef-3682-4750-a2e1-918ba6f16188,2017-04-07 15:14:00,NaT
1,0c3a3dd0-fb64-4eac-bf84-ba069ce409f2,2017-01-26 14:24:00,2017-01-26 14:44:00
2,6e0b2d60-4027-4d9a-babd-0e7d40859fb1,2017-08-20 08:23:00,2017-08-20 08:31:00
3,6879527e-c5a6-4d14-b2da-50b85212b0ab,2017-11-04 18:15:00,NaT
4,a84327ff-5daa-4ba1-b789-d5b4caf81e96,2017-02-27 11:25:00,NaT


### 4. Check length of `visits_cart`?

In [7]:
visits_cart_left.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   user_id     2000 non-null   object        
 1   visit_time  2000 non-null   datetime64[ns]
 2   cart_time   348 non-null    datetime64[ns]
dtypes: datetime64[ns](2), object(1)
memory usage: 47.0+ KB


### 5. How many timestamps are null in `cart_time`?

In [ ]:
cart_timestamp_null = visits_cart_left[visits_cart_left.cart_time.isnull()].user_id.count()

# display result
print('There are ' + str(cart_timestamp_null) + ' null values in timestamps column.')

There are 1652 null values in timestamps column.


### 6. What percentage only visited the webpage?

In [ ]:
percent_visits = visits_cart_left[visits_cart_left.cart_time.isnull()].user_id.count() * 100 / visits_cart_left.visit_time.count() 

# display result
print(str(percent_visits) + '% of customers only visited the webpage.')

82.6% of customers only visited the webpage.


### 7. What percentage placed a t-shirt in their cart but did not checkout?

In [ ]:
cart_checkout = pd.merge(cart, checkout, how='left')
percent_no_checkout = cart_checkout[cart_checkout.checkout_time.isnull()].user_id.count()/cart_checkout.user_id.count()

# display result
print('{:.2f}% of customers added items to their cart but did not complete the checkout process.'.format(percent_no_checkout * 100))

35.06% of customers added items to their cart but did not complete the checkout process.


### 8. Merge all four datasets

In [ ]:
all_tables = visits.merge(cart, how='left').merge(checkout, how='left').merge(purchase,how='left')

# inspect the first 5 rows of all_tables
all_tables.head(5)

,user_id,visit_time,cart_time,checkout_time,purchase_time
0,943647ef-3682-4750-a2e1-918ba6f16188,2017-04-07 15:14:00,NaT,NaT,NaT
1,0c3a3dd0-fb64-4eac-bf84-ba069ce409f2,2017-01-26 14:24:00,2017-01-26 14:44:00,2017-01-26 14:54:00,2017-01-26 15:08:00
2,6e0b2d60-4027-4d9a-babd-0e7d40859fb1,2017-08-20 08:23:00,2017-08-20 08:31:00,NaT,NaT
3,6879527e-c5a6-4d14-b2da-50b85212b0ab,2017-11-04 18:15:00,NaT,NaT,NaT
4,a84327ff-5daa-4ba1-b789-d5b4caf81e96,2017-02-27 11:25:00,NaT,NaT,NaT


### 9. Percentage of users who got to checkout but did not purchase

In [20]:
reached_checkout = all_tables[~all_tables.checkout_time.isnull()]
checkout_not_purchase = all_tables[(all_tables.purchase_time.isnull()) & (~all_tables.checkout_time.isnull())]
checkout_not_purchase_percent = float(len(checkout_not_purchase)) / float(len(reached_checkout))

# display result
print('{:.2f}% of customers reached the checkout page but did not complete the purchase.'.format(checkout_not_purchase_percent * 100))

24.55% of customers reached the checkout page but did not complete the purchase.


### Repeat step 9 using different approach

In [21]:
num_checkout = all_tables[all_tables.checkout_time.isnull() == False].user_id.count()
num_purchase = all_tables[all_tables.purchase_time.isnull() == False].user_id.count()
per_checkout_no_purchace = (num_checkout - num_purchase)  / num_checkout

# display result
print('{:.2f}% of customers reached the checkout page but did not complete the purchase.'.format(per_checkout_no_purchace * 100))  

24.55% of customers reached the checkout page but did not complete the purchase.


### 10. Check each part of the funnel

In [28]:
# display all calculated percentage for each funnel step
print('{:.2f}% of users who visited the site did not add an item to their cart.'.format(percent_visits))
print('{:.2f}% of users who added an item to their cart did not proceed to checkout.'.format(percent_no_checkout * 100))
print('{:.2f}% of users who checkedout an item did not complete the purchase.'.format(per_checkout_no_purchace * 100))


82.60% of users who visited the site did not add an item to their cart.
35.06% of users who added an item to their cart did not proceed to checkout.
24.55% of users who checkedout an item did not complete the purchase.


### Key Obersvation

*The weakest part of the funnel is clearly getting a person who visited the site to add a tshirt to their cart. Once they've added a t-shirt to their cart it is fairly likely they end up purchasing it. A suggestion could be to make the add-to-cart button more prominent on the front page.*


### 11. Estimate average time to purchase

In [35]:
# add new column 'time_to_purchase' to all_tables DataFrame

all_tables['time_to_purchase'] = all_tables.purchase_time - all_tables.visit_time

In [32]:
# inspect the first 5 rows of all_tables with the new column
all_tables.head(5)

,user_id,visit_time,cart_time,checkout_time,purchase_time,time_to_purchace
0,943647ef-3682-4750-a2e1-918ba6f16188,2017-04-07 15:14:00,NaT,NaT,NaT,NaT
1,0c3a3dd0-fb64-4eac-bf84-ba069ce409f2,2017-01-26 14:24:00,2017-01-26 14:44:00,2017-01-26 14:54:00,2017-01-26 15:08:00,0 days 00:44:00
2,6e0b2d60-4027-4d9a-babd-0e7d40859fb1,2017-08-20 08:23:00,2017-08-20 08:31:00,NaT,NaT,NaT
3,6879527e-c5a6-4d14-b2da-50b85212b0ab,2017-11-04 18:15:00,NaT,NaT,NaT,NaT
4,a84327ff-5daa-4ba1-b789-d5b4caf81e96,2017-02-27 11:25:00,NaT,NaT,NaT,NaT


In [38]:
# calculate the average time to purchase
avg_time_to_purchase = all_tables.time_to_purchase.mean()

In [46]:
# display result
print('The average time to purchase is: {}'.format((avg_time_to_purchase)))

The average time to purchase is: 0 days 00:43:12.380952
